# 05 - Keyword & Phrase Extraction

Extract the important keywords and phrases from customer feedback.

**Approach:** simple and explainable — TF-IDF + n-grams. No neural networks.

- **TF-IDF** scores which words are *rare and distinctive* (high importance)
- **n-grams** capture multi-word phrases like `payment gateway`, `app slow`

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.keyword_extractor import (
    extract_feedback_keywords,
    extract_phrases_ngrams,
    extract_keywords_with_index,
    build_keyword_index,
)
from src.data_loader import load_raw_tweets

## Method 1: local n-gram phrase extraction

In [ ]:
text = "The payment gateway failed during checkout."
print("Phrases:", extract_phrases_ngrams(text, n=2))
print("Keywords:", extract_feedback_keywords(text))

## Method 2: TF-IDF keyword extraction

We fit a TF-IDF model on the *whole corpus* first. Then for a single feedback, we look at which corpus-rare terms appear in it. A term like `gateway` is rare across feedback, so it ranks highly as a keyword.

In [ ]:
df = pd.DataFrame({"feedback": load_raw_tweets()["text"].astype(str)})
vectorizer, tfidf_matrix, feature_names = build_keyword_index(
    df, text_column="feedback"
)
print("Corpus vocab features:", tfidf_matrix.shape)

In [ ]:
examples = [
    "The payment gateway failed during checkout.",
    "The application is very slow and payment keeps failing.",
    "The support team solved my issue very quickly.",
    "I cannot log in to my account since yesterday.",
]

for ex in examples:
    kw = extract_keywords_with_index(ex, vectorizer, feature_names, top_n=5)
    print(f"{kw}")
    print(f"   <- {ex}")

## Why this works

1. `payment gateway` appears as a bigram and gets a high TF-IDF score because it is rare across the whole corpus.
2. Boring filler words (`the`, `is`, `and`) are filtered out by our stop-word list.
3. Negation words (`not`, `never`) are kept.

We never blindly return stop words.